# EN Augmentation Experiment — data preparation (upward alignment)
## 多语言混合训练版本

**Experiment goal:**
- Original training set: EN 446 + PL 674 + RU 684 = 1804 articles
- Augmented training set: EN **674** + PL 674 + RU 684 = 2032 articles
- 做法：Copy original training data and append 228 translated articles (FR/DE/IT → EN) with labels
- Dev set unchanged: EN 90 + PL 49 + RU 48

**Hypothesis:** Can translation diversity improve English detection performance (baseline Macro F1 = 41.76%)?

In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory: contains data/, results/, technique_list/, etc.
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# Training data directories
TRAIN_ARTICLES_DIR = "your/path/here"  # multilingual training articles
TRAIN_LABELS_FILE  = "your/path/here"  # corresponding label file
DEV_ARTICLES_DIR   = "your/path/here"  # dev articles
DEV_LABELS_FILE    = "your/path/here"  # dev labels

# SemEval-2023 dev data (per-language evaluation)
SEMEVAL_DEV_DIR = "your/path/here"  # contains en/, po/, ru/ subdirs

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


In [ ]:
import os
import sys
import shutil
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
from tqdm import tqdm

print('Imports loaded')

## Step 0: Path configuration
Update the paths below to match your local environment

In [ ]:
# ============================================================
# Path configuration
# ============================================================

ORIG_TRAIN_ARTICLES = "your/path/here"
ORIG_TRAIN_LABELS = "your/path/here"

DEV_ARTICLES = "your/path/here"
DEV_LABELS = "your/path/here"

TRANSLATED_ARTICLES = "your/path/here"
TRANSLATED_LABELS = "your/path/here"

OUTPUT_BASE = "your/path/here"
OUTPUT_ARTICLES = os.path.join(OUTPUT_BASE, "all_train_articles_en677")
OUTPUT_LABELS = os.path.join(OUTPUT_BASE, "all_train_articles_en677.txt")

EXPLANATIONS_FILE = "your/path/here"

TECHNIQUES_FILE = "your/path/here"
with open(TECHNIQUES_FILE, 'r', encoding='utf-8') as f:
    TECHNIQUES_23 = [line.strip() for line in f if line.strip()]
print(f"Loaded {len(TECHNIQUES_23)} techniques from {TECHNIQUES_FILE}")

print("Path config:")
print(f"  Orig train: {ORIG_TRAIN_ARTICLES}")
print(f"  Translated: {TRANSLATED_ARTICLES}")
print(f"  Output:     {OUTPUT_ARTICLES}")

## Step 0.5: Verify all paths exist

In [ ]:
print("Checking paths...")
all_ok = True
for path, desc in [
    (ORIG_TRAIN_ARTICLES, "原始训练文章"),
    (ORIG_TRAIN_LABELS, "原始训练标签"),
    (TRANSLATED_ARTICLES, "翻译文章(merged_en_677)"),
    (TRANSLATED_LABELS, "翻译标签(merged_labels_677)"),
    (DEV_ARTICLES, "Dev文章"),
    (DEV_LABELS, "Dev标签"),
]:
    exists = os.path.exists(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {desc}: {path}")
    if not exists:
        all_ok = False

if all_ok:
    print("\nAll paths exist, ready to proceed")
else:
    print("\nSome paths missing, please fix config above")

## Step 1: 分析原始训练数据格式
检查标签文件格式，确保新数据格式一致

In [ ]:
print("=" * 70)
print("STEP 1: Analyze original training data")
print("=" * 70)

orig_files = sorted(os.listdir(ORIG_TRAIN_ARTICLES))
orig_files = [f for f in orig_files if os.path.isfile(os.path.join(ORIG_TRAIN_ARTICLES, f))]
print(f"\n  Original articles: {len(orig_files)}")
print(f"  Examples: {orig_files[:3]}")

with open(ORIG_TRAIN_LABELS, 'r', encoding='utf-8', errors='replace') as f:
    orig_label_lines = [l.strip() for l in f.readlines() if l.strip()]

print(f"\n  Label lines: {len(orig_label_lines)}")
print(f"  First 5 lines:")
for line in orig_label_lines[:5]:
    print(f"    {line}")

col_counts = Counter()
article_ids = set()
for line in orig_label_lines:
    parts = line.split('\t')
    col_counts[len(parts)] += 1
    if parts:
        article_ids.add(parts[0])

print(f"\n  Column count dist: {dict(col_counts)}")
print(f"  Unique article_ids: {len(article_ids)}")

id_line_pairs = []
for line in orig_label_lines:
    parts = line.split('\t')
    if len(parts) >= 2:
        try:
            id_line_pairs.append((parts[0], int(parts[1])))
        except ValueError:
            pass

unique_pairs = len(set(id_line_pairs))
total_pairs = len(id_line_pairs)

if unique_pairs < total_pairs:
    label_format = "one_per_line"
    print(f"\n  Label format: ONE technique per line ({total_pairs} lines, {unique_pairs} unique pairs)")
else:
    label_format = "comma_separated"
    print(f"\n  Label format: COMMA-separated multi-technique ({unique_pairs} unique lines)")

print(f"  Detected: label_format = '{label_format}'")

## Step 2: Identify 228 translated articles
Distinguish the 446 original EN articles from the 228 translated ones in merged_en_674

In [ ]:
print("=" * 70)
print("STEP 2: Identify 228 translated articles")
print("=" * 70)

merged_files = sorted(os.listdir(TRANSLATED_ARTICLES))
merged_files = [f for f in merged_files if os.path.isfile(os.path.join(TRANSLATED_ARTICLES, f))]

translated_only = [f for f in merged_files if f.startswith(('fr_', 'ge_', 'it_'))]
original_en = [f for f in merged_files if not f.startswith(('fr_', 'ge_', 'it_'))]

print(f"  merged_en_677 total: {len(merged_files)}")
print(f"  Original EN: {len(original_en)} (already in training set)")
print(f"  Translated (FR/DE/IT→EN, 228 after QC): {len(translated_only)}")
print(f"    FR->EN: {len([f for f in translated_only if f.startswith('fr_')])}")
print(f"    DE->EN: {len([f for f in translated_only if f.startswith('ge_')])}")
print(f"    IT->EN: {len([f for f in translated_only if f.startswith('it_')])}")

if len(translated_only) == 228:
    print(f"  OK: 228 translated articles (FR/DE/IT → EN, QC-filtered)")
else:
    print(f"  WARNING: expected 228, got {len(translated_only)}")

orig_train_files = set(os.listdir(ORIG_TRAIN_ARTICLES))
conflicts = set(translated_only) & orig_train_files
if conflicts:
    print(f"  WARNING: {len(conflicts)} filename conflicts")
else:
    print(f"  OK: No filename conflicts")

## Step 3a: 创建增强文章文件夹
Copy original ~1804 articles + append 228 translated (QC-filtered)

In [ ]:
print("=" * 70)
print("STEP 3a: Create augmented article folder")
print("=" * 70)

print(f"  Output: {OUTPUT_ARTICLES}")

# Uncomment to rebuild:
# if os.path.exists(OUTPUT_ARTICLES):
#     shutil.rmtree(OUTPUT_ARTICLES)

if os.path.exists(OUTPUT_ARTICLES) and len(os.listdir(OUTPUT_ARTICLES)) > 0:
    total = len(os.listdir(OUTPUT_ARTICLES))
    print(f"  Already exists with {total} files. Uncomment rmtree above to rebuild.")
else:
    os.makedirs(OUTPUT_ARTICLES, exist_ok=True)
    
    orig_count = 0
    for fname in sorted(os.listdir(ORIG_TRAIN_ARTICLES)):
        src = os.path.join(ORIG_TRAIN_ARTICLES, fname)
        dst = os.path.join(OUTPUT_ARTICLES, fname)
        if os.path.isfile(src):
            try:
                os.symlink(os.path.abspath(src), dst)
            except OSError:
                shutil.copy2(src, dst)
            orig_count += 1
    print(f"  Original articles: {orig_count} linked")
    
    trans_count = 0
    for fname in translated_only:
        src = os.path.join(TRANSLATED_ARTICLES, fname)
        dst = os.path.join(OUTPUT_ARTICLES, fname)
        if os.path.isfile(src) and not os.path.exists(dst):
            try:
                os.symlink(os.path.abspath(src), dst)
            except OSError:
                shutil.copy2(src, dst)
            trans_count += 1
    print(f"  Translated articles: {trans_count} appended")
    
    total = len(os.listdir(OUTPUT_ARTICLES))
    print(f"  Total: {total} articles")

## Step 3b: 标签解析工具函数

In [ ]:
def parse_label_file(label_file, article_id):
    """Parse a single label file, return [(article_id, line_num, technique)]"""
    annotations = []
    try:
        with open(label_file, 'r', encoding='utf-8', errors='replace') as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]
    except Exception as e:
        print(f"  ERROR reading {label_file}: {e}")
        return []
    
    if not lines:
        return []
    
    for line in lines:
        parts = line.split('\t')
        if len(parts) >= 3:
            try:
                line_num = int(parts[1])
                for tech_field in parts[2:]:
                    for tech in tech_field.strip().split(','):
                        tech = tech.strip()
                        if tech and tech in TECHNIQUES_23:
                            annotations.append((article_id, line_num, tech))
            except ValueError:
                continue
        elif len(parts) == 2:
            try:
                line_num = int(parts[0])
                tech = parts[1].strip()
                if tech in TECHNIQUES_23:
                    annotations.append((article_id, line_num, tech))
            except ValueError:
                continue
    return annotations

print("parse_label_file() defined")

## Step 3c: 合并标签文件
原始标签 + 翻译文章标签 -> 一个文件

In [ ]:
# ===== 修复: 翻译文章标签路径 =====
# 真正的标签在 labels_en/ 目录，而非 merged_labels_677/
TRANSLATED_LABELS_CORRECT = "your/path/here"

# 构建 文章文件名 -> 标签文件路径 的映射
def find_label_for_translated(article_fname):
    """
    文章文件名: fr_article23100.txt
    标签文件名: fr_23100.txt (Step1中 replace('article','') 导致)
    """
    name = os.path.splitext(article_fname)[0]  # fr_article23100
    # 还原Step1的命名逻辑: 去掉 "article"
    label_name = name.replace('article', '') + '.txt'  # fr_23100.txt
    label_path = os.path.join(TRANSLATED_LABELS_CORRECT, label_name)
    if os.path.exists(label_path):
        return label_path
    return None

# 验证
found = 0
for fname in translated_only:
    path = find_label_for_translated(fname)
    if path:
        found += 1
print(f"匹配到标签: {found}/{len(translated_only)}")

# 看一个样本内容
sample = find_label_for_translated(sorted(translated_only)[0])
if sample:
    with open(sample) as f:
        print(f"\n样本标签内容:\n{f.read()[:500]}")

In [ ]:
# ===== 修复后的 Step 3c =====
TRANSLATED_LABELS_CORRECT = "your/path/here"

print("=" * 70)
print("STEP 3c: Create consolidated label file (FIXED)")
print("=" * 70)

# 1. 读取原始标签
with open(ORIG_TRAIN_LABELS, 'r', encoding='utf-8', errors='replace') as f:
    orig_labels_content = f.read()
orig_line_count = len([l for l in orig_labels_content.strip().split('\n') if l.strip()])
print(f"  Original labels: {orig_line_count} lines")

# 2. 处理翻译文章的标签
translated_label_lines = []
trans_labels_found = 0
trans_labels_empty = 0
trans_technique_counts = Counter()

for fname in translated_only:
    # 文章文件名: fr_article23100.txt -> 目标key: fr_article23100
    article_key = os.path.splitext(fname)[0]  # fr_article23100
    
    # 标签文件名: fr_23100.txt (Step1中去掉了'article')
    label_name = article_key.replace('article', '') + '.txt'
    label_path = os.path.join(TRANSLATED_LABELS_CORRECT, label_name)
    
    if not os.path.exists(label_path):
        trans_labels_empty += 1
        continue
    
    with open(label_path, 'r', encoding='utf-8') as f:
        lines = [l.strip() for l in f if l.strip()]
    
    if not lines:
        trans_labels_empty += 1
        continue
    
    trans_labels_found += 1
    
    for line in lines:
        parts = line.split('\t')
        if len(parts) >= 3:
            orig_id = parts[0]      # "23100"
            line_num = parts[1]     # "1"
            techniques = parts[2]   # "Doubt,Name_Calling-Labeling"
            
            # 用完整的 article_key 替换原始 id
            translated_label_lines.append(f"{article_key}\t{line_num}\t{techniques}")
            
            for tech in techniques.split(','):
                tech = tech.strip()
                if tech:
                    trans_technique_counts[tech] += 1

print(f"  Translated labels: {trans_labels_found} with labels, {trans_labels_empty} empty")
print(f"  New label lines: {len(translated_label_lines)}")
if trans_technique_counts:
    print(f"  Top 5 techniques:")
    for tech, count in trans_technique_counts.most_common(5):
        print(f"    {tech}: {count}")

# 3. 合并写入
with open(OUTPUT_LABELS, 'w', encoding='utf-8') as f:
    f.write(orig_labels_content)
    if not orig_labels_content.endswith('\n'):
        f.write('\n')
    f.write('\n'.join(translated_label_lines) + '\n')

total_lines = orig_line_count + len(translated_label_lines)
print(f"\n  Summary: {orig_line_count} orig + {len(translated_label_lines)} new = {total_lines} total")
print(f"  Saved: {OUTPUT_LABELS}")

## Step 4: 验证增强数据集

In [ ]:
print("=" * 70)
print("STEP 4: Validate augmented dataset")
print("=" * 70)

all_files = sorted(os.listdir(OUTPUT_ARTICLES))
all_files = [f for f in all_files 
             if os.path.isfile(os.path.join(OUTPUT_ARTICLES, f)) 
             or os.path.islink(os.path.join(OUTPUT_ARTICLES, f))]

en_trans = [f for f in all_files if f.startswith(('fr_', 'ge_', 'it_'))]

print(f"  Total articles: {len(all_files)}")
print(f"  Original (EN+PL+RU): {len(all_files) - len(en_trans)}")
print(f"  New translated (->EN): {len(en_trans)}")

with open(OUTPUT_LABELS, 'r', encoding='utf-8', errors='replace') as f:
    label_lines = [l.strip() for l in f.readlines() if l.strip()]

label_ids = set()
for line in label_lines:
    parts = line.split('\t')
    if parts:
        label_ids.add(parts[0])

trans_ids = set(os.path.splitext(f)[0] for f in en_trans)
trans_in_labels = trans_ids & label_ids

print(f"  Label lines: {len(label_lines)}")
print(f"  Translated in labels: {len(trans_in_labels)}/{len(trans_ids)}")

dev_count = len([f for f in os.listdir(DEV_ARTICLES) if os.path.isfile(os.path.join(DEV_ARTICLES, f))])
print(f"  Dev set: {dev_count} articles (unchanged)")

if len(en_trans) == 228 and len(trans_in_labels) >= 200:
    print(f"\n  VALIDATION PASSED!")
else:
    print(f"\n  ISSUES FOUND - please check above")

## Step 5: 快速测试数据加载

In [ ]:
def test_make_dataframe(data_type='train'):
    if data_type == 'train':
        input_folder = OUTPUT_ARTICLES + '/'
        labels_fn = OUTPUT_LABELS
    elif data_type == 'dev':
        input_folder = DEV_ARTICLES + '/'
        labels_fn = DEV_LABELS
    else:
        raise ValueError("data_type must be 'train' or 'dev'")
    
    text = []
    skipped = 0
    for fil in tqdm(filter(lambda x: x.endswith('.txt'), os.listdir(input_folder))):
        iD = fil.split('.')[0]
        try:
            with open(os.path.join(input_folder, fil), 'r', encoding='utf-8') as f:
                content = f.read()
            lines = list(enumerate(content.splitlines(), 1))
            text.extend([(iD,) + line for line in lines])
        except UnicodeDecodeError:
            skipped += 1
    if skipped:
        print(f"Skipped {skipped} files")
    
    df_text = pd.DataFrame(text, columns=['id','line','text'])
    df_text.line = df_text.line.apply(int)
    df_text = df_text[df_text.text.str.strip().str.len() > 0].copy()
    df_text = df_text.set_index(['id','line'])
    
    labels_data = []
    with open(labels_fn, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split('\t')
            if len(parts) >= 2:
                try:
                    labels_data.append([parts[0], int(parts[1]), parts[2] if len(parts) >= 3 else ''])
                except ValueError:
                    pass
    
    if labels_data:
        labels = pd.DataFrame(labels_data, columns=['id', 'line', 'labels'])
        labels['line'] = labels['line'].astype(int)
        labels = labels.set_index(['id','line'])
        df = df_text.join(labels, how='left')[['text','labels']]
        df['labels'] = df['labels'].fillna('')
    else:
        df = df_text.copy()
        df['labels'] = ''
        df = df[['text', 'labels']]
    
    return df.reset_index()

print("Loading augmented train set...")
df_train = test_make_dataframe('train')
print(f"  Train: {len(df_train)} paragraphs, {df_train['id'].nunique()} articles")
print(f"  With labels: {(df_train['labels'] != '').sum()}")

trans_ids = [os.path.splitext(f)[0] for f in en_trans]
loaded_trans = df_train[df_train['id'].isin(trans_ids)]
print(f"  Translated paragraphs: {len(loaded_trans)}, with labels: {(loaded_trans['labels'] != '').sum()}")

print(f"\nLoading dev set...")
df_dev = test_make_dataframe('dev')
print(f"  Dev: {len(df_dev)} paragraphs, {df_dev['id'].nunique()} articles")

print(f"\nData loading test PASSED!")

## Step 6: 训练代码修改

### 只需要改3处：

```python
# 修改1: make_dataframe() 中的训练数据路径
if data_type == 'train':
    input_folder = "your/path/here"
    labels_fn = "your/path/here"
# dev路径不变!

# 修改2: S3前缀
S3_PREFIX = "multi_label_models/en_augmentation_677"

# 修改3: checkpoint文件名
filename = f"{self.model_name.split('/')[-1]}_{EPOCHS}_{LOSS_TYPE}_en677_dual_encoder.ckpt"
```

### 所有超参数不变！

| 参数 | 值 |
|------|-----|
| model | xlm-roberta-base |
| epochs | 10 |
| batch | 8 x 4 = 32 |
| max_length | 256 |
| lr | 1e-5 |
| warmup | 1000 |
| loss | bce |

## 训练完成后：预期结果对比表

| 条件 | 训练数据 | EN Macro F1 | PL Macro F1 | RU Macro F1 |
|------|---------|-------------|-------------|-------------|
| **原始** | EN 446 + PL 674 + RU 684 | 41.76% | 52.23% | 39.57% |
| **EN+141** | EN **674** + PL 674 + RU 684 (34% trans.) | **36.96%** | **52.23%** | **39.57%** |

**关键观察：**
- EN Macro F1 > 41.76% → 翻译多样性对EN有效
- PL/RU基本不变 → 变化确实来自EN数据增加
- 关注EN的Loaded_Language FP占比是否从48.7%下降

**训练完成后把三个语言的结果告诉我，我帮你分析并更新rebuttal。**